In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Open the notebook from the project or experiments folder.")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import load_training_config
from src.models.train import load_training_data

config = load_training_config()
portfolio = load_training_data(config)

assert portfolio["partition"].eq("train").all()
assert portfolio["customer_id"].is_unique

print(f"Exploration customers: {len(portfolio):,}")

c:\Users\antek\Customer_Chunrn_End_to_End\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Exploration customers: 700,000


In [2]:
charges = portfolio["monthlycharges"]

display(
    charges.describe(
        percentiles=[0.01, 0.10, 0.50, 0.90, 0.95, 0.99, 0.999]
    ).to_frame("monthlycharges")
)

px.histogram(
    portfolio,
    x="monthlycharges",
    nbins=100,
    title="Monthly charges: full distribution, including extreme values",
    labels={"monthlycharges": "Monthly charges"},
).show()

,monthlycharges
count,700000.000000
mean,86.428309
std,27.563500
min,20.000000
1%,36.910000
10%,56.570000
50%,85.470000
90%,115.520000
95%,125.150000
99%,145.180000


In [3]:
revenue_proxy = float(charges.sum())

if revenue_proxy <= 0:
    raise ValueError("Revenue concentration requires a positive total.")

ranked = portfolio.sort_values(
    ["monthlycharges", "customer_id"],
    ascending=[False, True],
)

rows = []

for fraction in (0.01, 0.05, 0.10, 0.20):
    count = int(np.ceil(fraction * len(ranked)))
    group = ranked.head(count)

    rows.append({
        "top_customer_fraction": fraction,
        "customers": count,
        "monthly_charge_share": group["monthlycharges"].sum() / revenue_proxy,
        "mean_monthly_charge": group["monthlycharges"].mean(),
    })

concentration = pd.DataFrame(rows)
display(concentration)

px.bar(
    concentration.assign(
        customer_group=lambda frame: (
            "Top " + (100 * frame["top_customer_fraction"]).astype(int).astype(str)
            + "%"
        )
    ),
    x="customer_group",
    y="monthly_charge_share",
    title="Share of total monthly charges by highest-charge customers",
    labels={
        "customer_group": "Customer group",
        "monthly_charge_share": "Share of monthly charges",
    },
).update_yaxes(tickformat=".1%").show()

,top_customer_fraction,customers,monthly_charge_share,mean_monthly_charge
0,0.01,7000,0.022329,192.983144
1,0.05,35000,0.083798,144.849865
2,0.10,70000,0.153140,132.356186
3,0.20,140000,0.279919,120.964571


In [5]:
q50, q90, q99 = charges.quantile([0.50, 0.90, 0.99])

thresholds = pd.Series({
    "median": q50,
    "90th_percentile": q90,
    "99th_percentile": q99,
})

display(thresholds.to_frame("monthly_charge_threshold"))

portfolio["revenue_segment"] = pd.cut(
    charges,
    bins=[-np.inf, q50, q90, q99, np.inf],
    labels=[
        "Lower half",
        "Middle 40%",
        "High revenue",
        "Very high revenue",
    ],
    right=True,
)

segments = portfolio.groupby(
    "revenue_segment", observed=True
).agg(
    customers=("customer_id", "size"),
    monthly_charge_total=("monthlycharges", "sum"),
    mean_monthly_charge=("monthlycharges", "mean"),
    median_tenure=("tenure", "median"),
    observed_churn_rate=("churn", "mean"),
    mean_satisfaction=("customer_satisfaction", "mean"),
    satisfaction_coverage=("customer_satisfaction", "count"),
    mean_complaints=("num_complaints", "mean"),
    complaints_coverage=("num_complaints", "count"),
)

segments["customer_share"] = segments["customers"] / len(portfolio)
segments["monthly_charge_share"] = (
    segments["monthly_charge_total"] / revenue_proxy
)

assert portfolio["revenue_segment"].notna().all()
assert segments["customers"].sum() == len(portfolio)
assert np.isclose(segments["monthly_charge_share"].sum(), 1)

display(segments)

,monthly_charge_threshold
median,85.47
90th_percentile,115.52
99th_percentile,145.18


,customers,monthly_charge_total,mean_monthly_charge,median_tenure,observed_churn_rate,mean_satisfaction,satisfaction_coverage,mean_complaints,complaints_coverage,customer_share,monthly_charge_share
revenue_segment,,,,,,,,,,,
Lower half,350044,2.375633e+07,67.866709,16.0,0.101667,6.161575,343210,0.699626,339530,0.500063,0.392668
Middle 40%,279992,2.748271e+07,98.155333,16.0,0.099114,6.154372,274338,0.701185,271616,0.399989,0.454261
High revenue,62966,7.910183e+06,125.626253,16.0,0.088095,6.162096,61667,0.702085,61145,0.089951,0.130747
Very high revenue,6998,1.350592e+06,192.996806,16.0,0.081881,6.154698,6865,0.706453,6803,0.009997,0.022324


In [6]:
contract_mix = pd.crosstab(
    portfolio["revenue_segment"],
    portfolio["contract"],
    normalize="index",
)

display(contract_mix)

px.bar(
    contract_mix.reset_index().melt(
        id_vars="revenue_segment",
        var_name="contract",
        value_name="customer_share",
    ),
    x="revenue_segment",
    y="customer_share",
    color="contract",
    barmode="stack",
    title="Contract composition within revenue segments",
).update_yaxes(tickformat=".0%").show()

contract,month_to_month,one_year,two_year
revenue_segment,,,
Lower half,0.020126,0.549551,0.430323
Middle 40%,0.019608,0.551830,0.428562
High revenue,0.019534,0.552473,0.427993
Very high revenue,0.019863,0.553730,0.426408


In [7]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from src.models.artifacts import current_provenance, read_json
from src.models.pipeline import FEATURE_COLUMNS

FINALIST_RUN_ID = "4b8fa8456a984b8da19bb09e4a5d2334"

database = Path(config.mlflow.tracking_directory) / "mlflow.db"
if not database.is_file():
    raise FileNotFoundError(database)

previous_uri = mlflow.get_tracking_uri()
mlflow.set_tracking_uri(f"sqlite:///{database.as_posix()}")

try:
    client = MlflowClient()
    source = client.get_run(FINALIST_RUN_ID)

    if (
        source.info.status != "FINISHED"
        or source.data.tags.get("model") != "catboost"
    ):
        raise ValueError("Expected the completed CatBoost finalist.")

    parent_id = source.data.tags["mlflow.parentRunId"]
    saved = read_json(client, parent_id, "input_provenance.json")
    current = current_provenance(config)

    if any(
        current[name]["sha256"] != saved[name]["sha256"]
        for name in current
    ):
        raise ValueError("Portfolio inputs differ from the finalist inputs.")

    comparisons = read_json(client, parent_id, "validation_comparison.json")
    matches = [
        row for row in comparisons
        if row["run_id"] == FINALIST_RUN_ID
    ]

    if len(matches) != 1 or not matches[0]["model_uri"]:
        raise ValueError("Expected one saved CatBoost model.")

    # Load only trusted, locally generated artifacts.
    model = mlflow.sklearn.load_model(matches[0]["model_uri"])

    positive = np.flatnonzero(model.classes_ == 1)
    if len(positive) != 1:
        raise ValueError("Expected one positive churn class.")

    probabilities = model.predict_proba(
        portfolio.loc[:, list(FEATURE_COLUMNS)]
    )[:, positive[0]]

    if (
        probabilities.shape != (len(portfolio),)
        or not np.isfinite(probabilities).all()
        or (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise ValueError("Invalid churn probabilities.")

    portfolio["churn_probability"] = probabilities

finally:
    mlflow.set_tracking_uri(previous_uri)

In [8]:
risk_q50, risk_q90 = portfolio["churn_probability"].quantile([0.5, 0.9])

if risk_q50 >= risk_q90:
    raise ValueError("Risk thresholds are not distinct.")

display(pd.Series({
    "risk_50th_percentile": risk_q50,
    "risk_90th_percentile": risk_q90,
}).to_frame("probability_threshold"))

portfolio["risk_segment"] = pd.cut(
    portfolio["churn_probability"],
    bins=[-np.inf, risk_q50, risk_q90, np.inf],
    labels=["Lower risk", "Elevated risk", "Highest risk"],
    right=True,
)

portfolio["risk_weighted_monthly_charges"] = (
    portfolio["monthlycharges"] * portfolio["churn_probability"]
)

,probability_threshold
risk_50th_percentile,0.085126
risk_90th_percentile,0.178536


In [9]:
matrix = portfolio.groupby(
    ["revenue_segment", "risk_segment"],
    observed=True,
).agg(
    customers=("customer_id", "size"),
    monthly_charge_total=("monthlycharges", "sum"),
    mean_monthly_charge=("monthlycharges", "mean"),
    mean_predicted_churn=("churn_probability", "mean"),
    risk_weighted_monthly_charges=("risk_weighted_monthly_charges", "sum"),
)

matrix["customer_share"] = matrix["customers"] / len(portfolio)
matrix["monthly_charge_share"] = (
    matrix["monthly_charge_total"] / portfolio["monthlycharges"].sum()
)

assert portfolio["risk_segment"].notna().all()
assert matrix["customers"].sum() == len(portfolio)
assert np.isclose(
    matrix["monthly_charge_total"].sum(),
    portfolio["monthlycharges"].sum(),
)

display(matrix)

customers  monthly_charge_total  \
revenue_segment   risk_segment                                     
Lower half        Lower risk        164817          1.130857e+07   
                  Elevated risk     149713          1.006881e+07   
                  Highest risk       35514          2.378957e+06   
Middle 40%        Lower risk        143431          1.412820e+07   
                  Elevated risk     107334          1.050267e+07   
                  Highest risk       29227          2.851834e+06   
High revenue      Lower risk         37314          4.697085e+06   
                  Elevated risk      20815          2.609101e+06   
                  Highest risk        4837          6.039962e+05   
Very high revenue Lower risk          4438          8.194294e+05   
                  Elevated risk       2138          4.325047e+05   
                  Highest risk         422          9.865759e+04   

                                 mean_monthly_charge  mean_predicted_churn  \
revenue_segment   risk_segment                                               
Lower half        Lower risk               68.612863              0.054996   
                  Elevated risk            67.254089              0.121448   
                  Highest risk             66.986443              0.238516   
Middle 40%        Lower risk               98.501731              0.053421   
                  Elevated risk            97.850377              0.119455   
                  Highest risk             97.575324              0.242415   
High revenue      Lower risk              125.879966              0.051809   
                  Elevated risk           125.347170              0.117899   
                  Highest risk            124.870012              0.237516   
Very high revenue Lower risk              184.639331              0.050446   
                  Elevated risk           202.294063              0.117969   
                  Highest risk            233.785756              0.234409   

                                 risk_weighted_monthly_charges  \
revenue_segment   risk_segment                                   
Lower half        Lower risk                      6.200700e+05   
                  Elevated risk                   1.219765e+06   
                  Highest risk                    5.681206e+05   
Middle 40%        Lower risk                      7.542585e+05   
                  Elevated risk                   1.254239e+06   
                  Highest risk                    6.910124e+05   
High revenue      Lower risk                      2.432234e+05   
                  Elevated risk                   3.075554e+05   
                  Highest risk                    1.434266e+05   
Very high revenue Lower risk                      4.192468e+04   
                  Elevated risk                   5.133484e+04   
                  Highest risk                    2.370167e+04   

                                 customer_share  monthly_charge_share  
revenue_segment   risk_segment                                         
Lower half        Lower risk           0.235453              0.186919  
                  Elevated risk        0.213876              0.166427  
                  Highest risk         0.050734              0.039322  
Middle 40%        Lower risk           0.204901              0.233525  
                  Elevated risk        0.153334              0.173598  
                  Highest risk         0.041753              0.047138  
High revenue      Lower risk           0.053306              0.077638  
                  Elevated risk        0.029736              0.043126  
                  Highest risk         0.006910              0.009983  
Very high revenue Lower risk           0.006340              0.013544  
                  Elevated risk        0.003054              0.007149  
                  Highest risk         0.000603              0.001631

In [10]:
revenue_order = portfolio["revenue_segment"].cat.categories
risk_order = portfolio["risk_segment"].cat.categories

for metric, title, number_format in [
    ("customers", "Customers by revenue and predicted risk", ",.0f"),
    (
        "risk_weighted_monthly_charges",
        "Model-based monthly-charge exposure by segment",
        ",.0f",
    ),
]:
    grid = (
        matrix[metric]
        .unstack("risk_segment")
        .reindex(index=revenue_order, columns=risk_order)
        .fillna(0)
    )

    px.imshow(
        grid,
        text_auto=number_format,
        aspect="auto",
        color_continuous_scale="Blues",
        labels={
            "x": "Relative predicted risk",
            "y": "Revenue segment",
            "color": metric,
        },
        title=title,
    ).show()

In [11]:
from dataclasses import asdict

from src.models.campaign import CampaignScenario, compare_policies

scenario = CampaignScenario(
    contact_cost=2.0,
    gross_margin=0.2,
    retained_months=6.0,
    prevention_probability=0.2,
    capacity_fraction=0.1,
)

display(pd.Series(asdict(scenario)).to_frame("assumption"))

scored, policy_summary = compare_policies(
    portfolio[["customer_id", "monthlycharges"]],
    portfolio["churn_probability"].to_numpy(),
    scenario,
)

capacity = int(np.floor(scenario.capacity_fraction * len(portfolio)))

selected_ids = (
    scored.loc[scored["expected_net_value"] > 0]
    .sort_values(
        ["expected_net_value", "customer_id"],
        ascending=[False, True],
    )
    .head(capacity)["customer_id"]
)

campaign_portfolio = portfolio[
    ["customer_id", "revenue_segment", "risk_segment"]
].merge(
    scored,
    on="customer_id",
    validate="one_to_one",
)

campaign_portfolio["eligible"] = (
    campaign_portfolio["expected_net_value"] > 0
)
campaign_portfolio["selected"] = (
    campaign_portfolio["customer_id"].isin(selected_ids)
)

campaign_portfolio["eligible_expected_net_value"] = (
    campaign_portfolio["expected_net_value"]
    .where(campaign_portfolio["eligible"], 0.0)
)
campaign_portfolio["selected_expected_net_value"] = (
    campaign_portfolio["expected_net_value"]
    .where(campaign_portfolio["selected"], 0.0)
)

display(policy_summary)

,assumption
contact_cost,2.0
gross_margin,0.2
retained_months,6.0
prevention_probability,0.2
capacity_fraction,0.1


,policy,capacity_customers,selected_customers,selected_fraction,unused_capacity,expected_churners_contacted,expected_prevented_churn,expected_retained_contribution,campaign_cost,expected_net_value,scenario_roi
0,no_contact,70000,0,0.0,70000,0.00000,0.000000,0.000000,0.0,0.000000,NaN
1,risk_positive_only,70000,70000,0.1,0,16773.34784,3354.669568,345073.619955,140000.0,205073.619955,1.464812
2,expected_profit_positive_only,70000,70000,0.1,0,15731.23464,3146.246928,372102.686101,140000.0,232102.686101,1.657876


In [12]:
campaign_matrix = campaign_portfolio.groupby(
    ["revenue_segment", "risk_segment"],
    observed=True,
).agg(
    customers=("customer_id", "size"),
    eligible_customers=("eligible", "sum"),
    selected_customers=("selected", "sum"),
    mean_expected_net_value=("expected_net_value", "mean"),
    eligible_expected_net_value=("eligible_expected_net_value", "sum"),
    selected_expected_net_value=("selected_expected_net_value", "sum"),
)

campaign_matrix["eligible_fraction"] = (
    campaign_matrix["eligible_customers"] / campaign_matrix["customers"]
)
campaign_matrix["selected_fraction"] = (
    campaign_matrix["selected_customers"] / campaign_matrix["customers"]
)

profit_policy = policy_summary.set_index("policy").loc[
    "expected_profit_positive_only"
]

assert campaign_matrix["customers"].sum() == len(portfolio)
assert (
    campaign_matrix["selected_customers"].sum()
    == profit_policy["selected_customers"]
)
assert np.isclose(
    campaign_matrix["selected_expected_net_value"].sum(),
    profit_policy["expected_net_value"],
)

display(campaign_matrix)

customers  eligible_customers  \
revenue_segment   risk_segment                                   
Lower half        Lower risk        164817                   0   
                  Elevated risk     149713               63276   
                  Highest risk       35514               34113   
Middle 40%        Lower risk        143431                6600   
                  Elevated risk     107334              100548   
                  Highest risk       29227               29227   
High revenue      Lower risk         37314               10234   
                  Elevated risk      20815               20815   
                  Highest risk        4837                4837   
Very high revenue Lower risk          4438                2140   
                  Elevated risk       2138                2138   
                  Highest risk         422                 422   

                                 selected_customers  mean_expected_net_value  \
revenue_segment   risk_segment                                                 
Lower half        Lower risk                      0                -1.097079   
                  Elevated risk                   0                -0.044635   
                  Highest risk                16125                 1.839301   
Middle 40%        Lower risk                      0                -0.737916   
                  Elevated risk               10372                 0.804492   
                  Highest risk                29154                 3.674307   
High revenue      Lower risk                      0                -0.435611   
                  Elevated risk                7305                 1.546159   
                  Highest risk                 4837                 5.116472   
Very high revenue Lower risk                    273                 0.267220   
                  Elevated risk                1512                 3.762564   
                  Highest risk                  422                11.479624   

                                 eligible_expected_net_value  \
revenue_segment   risk_segment                                 
Lower half        Lower risk                        0.000000   
                  Elevated risk                 29823.953700   
                  Highest risk                  65700.528940   
Middle 40%        Lower risk                      726.833480   
                  Elevated risk                 86945.328107   
                  Highest risk                 107388.968562   
High revenue      Lower risk                     3014.983527   
                  Elevated risk                 32183.295172   
                  Highest risk                  24748.374082   
Very high revenue Lower risk                     2750.243284   
                  Elevated risk                  8044.361600   
                  Highest risk                   4844.401239   

                                 selected_expected_net_value  \
revenue_segment   risk_segment                                 
Lower half        Lower risk                        0.000000   
                  Elevated risk                     0.000000   
                  Highest risk                  47046.217088   
Middle 40%        Lower risk                        0.000000   
                  Elevated risk                 21502.691927   
                  Highest risk                 107263.785789   
High revenue      Lower risk                        0.000000   
                  Elevated risk                 17857.376625   
                  Highest risk                  24748.374082   
Very high revenue Lower risk                     1674.992927   
                  Elevated risk                  7164.846424   
                  Highest risk                   4844.401239   

                                 eligible_fraction  selected_fraction  
revenue_segment   risk_segment                                         
Lower half        Lower risk              0.00000

In [13]:
for metric, title in [
    (
        "selected_customers",
        "Profit-policy selected customers by revenue and risk",
    ),
    (
        "selected_expected_net_value",
        "Selected audience: scenario-based expected net value",
    ),
]:
    grid = (
        campaign_matrix[metric]
        .unstack("risk_segment")
        .reindex(
            index=portfolio["revenue_segment"].cat.categories,
            columns=portfolio["risk_segment"].cat.categories,
        )
        .fillna(0)
    )

    px.imshow(
        grid,
        text_auto=",.0f",
        aspect="auto",
        color_continuous_scale="Blues",
        labels={
            "x": "Relative predicted risk",
            "y": "Revenue segment",
            "color": metric,
        },
        title=title,
    ).show()

In [14]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

CLUSTER_COLUMNS = [
    "monthlycharges",
    "tenure",
    "avg_monthly_gb",
    "customer_satisfaction",
    "num_complaints",
    "num_service_calls",
    "late_payments",
    "days_since_last_interaction",
]

LOG_COLUMNS = [
    name for name in CLUSTER_COLUMNS
    if name != "customer_satisfaction"
]


def clustering_inputs(frame):
    features = frame.loc[:, CLUSTER_COLUMNS].astype(float).copy()
    values = features.to_numpy()

    if np.isinf(values).any() or (values < 0).any():
        raise ValueError(
            "Clustering features must be nonnegative and finite or missing."
        )

    features[LOG_COLUMNS] = np.log1p(features[LOG_COLUMNS])
    return features

In [15]:
if (
    len(portfolio) < 65_000
    or not portfolio["partition"].eq("train").all()
):
    raise ValueError("Expected at least 65,000 training-only rows.")

sample = portfolio.sample(n=65_000, random_state=42)

sample_a = sample.iloc[:30_000]
sample_b = sample.iloc[30_000:60_000]
probe = sample.iloc[60_000:].copy()

transformed_probe = clustering_inputs(probe)

results = []
candidates = {}

with threadpool_limits(limits=2):
    for k in (3, 4, 5):
        labels = []
        scores = []
        smallest_shares = []

        for name, fit_data, seed in (
            ("reference", sample_a, 42),
            ("new_seed", sample_a, 123),
            ("new_sample", sample_b, 42),
        ):
            model = make_pipeline(
                SimpleImputer(
                    strategy="median",
                    keep_empty_features=True,
                ),
                StandardScaler(),
                MiniBatchKMeans(
                    n_clusters=k,
                    n_init=5,
                    batch_size=2048,
                    max_iter=100,
                    random_state=seed,
                ),
            )

            model.fit(clustering_inputs(fit_data))
            predicted = model.predict(transformed_probe)

            if len(np.unique(predicted)) != k:
                raise ValueError(
                    f"k={k}: not all clusters represented in probe."
                )

            space = model[:-1].transform(transformed_probe)

            scores.append(
                silhouette_score(
                    space,
                    predicted,
                    sample_size=2000,
                    random_state=42,
                )
            )
            labels.append(predicted)
            smallest_shares.append(
                np.bincount(predicted, minlength=k).min() / len(probe)
            )

            if name == "reference":
                candidates[k] = model

        results.append({
            "k": k,
            "mean_probe_silhouette": np.mean(scores),
            "initialization_ARI": adjusted_rand_score(
                labels[0], labels[1]
            ),
            "sample_ARI": adjusted_rand_score(
                labels[0], labels[2]
            ),
            "smallest_probe_cluster_share": min(smallest_shares),
        })

cluster_results = pd.DataFrame(results)
display(cluster_results)

,k,mean_probe_silhouette,initialization_ARI,sample_ARI,smallest_probe_cluster_share
0,3,0.129293,0.243275,0.196828,0.1448
1,4,0.130205,0.490754,0.803555,0.1398
2,5,0.111531,0.542861,0.565446,0.1380


In [17]:
cluster_model = candidates[4]

cluster_profile = probe.copy()
cluster_profile["cluster"] = cluster_model.predict(
    clustering_inputs(cluster_profile)
)

cluster_sizes = (
    cluster_profile.groupby("cluster")
    .size()
    .to_frame("customers")
)
cluster_sizes["customer_share"] = (
    cluster_sizes["customers"] / len(cluster_profile)
)

cluster_means = cluster_profile.groupby("cluster")[CLUSTER_COLUMNS].mean()
cluster_medians = cluster_profile.groupby("cluster")[CLUSTER_COLUMNS].median()

display(cluster_sizes)
display(cluster_means)
display(cluster_medians)

,customers,customer_share
cluster,,
0,1263,0.2526
1,1618,0.3236
2,1420,0.2840
3,699,0.1398


,monthlycharges,tenure,avg_monthly_gb,customer_satisfaction,num_complaints,num_service_calls,late_payments,days_since_last_interaction
cluster,,,,,,,,
0,90.562090,23.064133,47.173713,6.193994,0.612161,1.555819,1.197941,46.095804
1,92.174205,20.848578,47.095633,6.022138,0.133878,1.040791,0.000000,44.132880
2,90.456871,23.523239,45.176638,6.285303,1.499259,2.755634,0.073239,44.128169
3,59.229986,23.402003,0.285172,6.286131,0.637574,1.732475,0.373391,45.170243


,monthlycharges,tenure,avg_monthly_gb,customer_satisfaction,num_complaints,num_service_calls,late_payments,days_since_last_interaction
cluster,,,,,,,,
0,89.300,16.0,33.540,7.0,0.0,1.0,1.0,31.0
1,89.820,14.0,34.080,6.0,0.0,1.0,0.0,29.5
2,88.985,18.0,32.025,7.0,1.0,3.0,0.0,31.0
3,56.890,17.0,0.000,7.0,0.0,1.0,0.0,31.0


In [19]:
scaled_probe = pd.DataFrame(
    cluster_model[:-1].transform(
        clustering_inputs(cluster_profile)
    ),
    columns=CLUSTER_COLUMNS,
    index=cluster_profile.index,
)

relative_profiles = (
    scaled_probe.groupby(cluster_profile["cluster"]).mean()
    - scaled_probe.mean()
)

px.imshow(
    relative_profiles,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Cluster feature deviations",
    labels={
        "x": "Clustering feature",
        "y": "Exploratory cluster",
        "color": "Scaled deviation",
    },
).show()

In [20]:
missingness_by_cluster = (
    cluster_profile[CLUSTER_COLUMNS]
    .isna()
    .groupby(cluster_profile["cluster"])
    .mean()
)

display(missingness_by_cluster)

,monthlycharges,tenure,avg_monthly_gb,customer_satisfaction,num_complaints,num_service_calls,late_payments,days_since_last_interaction
cluster,,,,,,,,
0,0.0,0.0,0.064925,0.024545,0.036421,0.0,0.0,0.0
1,0.0,0.0,0.053152,0.022868,0.016687,0.0,0.0,0.0
2,0.0,0.0,0.063380,0.022535,0.049296,0.0,0.0,0.0
3,0.0,0.0,0.004292,0.020029,0.032904,0.0,0.0,0.0


In [21]:
for segment in ("revenue_segment", "risk_segment"):
    display(
        pd.crosstab(
            cluster_profile["cluster"],
            cluster_profile[segment],
            normalize="index",
        )
    )

revenue_segment,Lower half,Middle 40%,High revenue,Very high revenue
cluster,,,,
0,0.422011,0.454473,0.112431,0.011085
1,0.420272,0.447466,0.121137,0.011125
2,0.440141,0.457042,0.092254,0.010563
3,0.922747,0.077253,0.000000,0.000000


risk_segment,Lower risk,Elevated risk,Highest risk
cluster,,,
0,0.414885,0.463183,0.121932
1,0.663164,0.305315,0.031520
2,0.402113,0.475352,0.122535
3,0.453505,0.437768,0.108727


In [22]:
internet_counts = pd.crosstab(
    cluster_profile["cluster"],
    cluster_profile["has_internet_service"],
).reindex(columns=[0, 1], fill_value=0)

internet_counts.columns = ["No internet", "Has internet"]

internet_shares = internet_counts.div(
    internet_counts.sum(axis=1),
    axis=0,
)

display(internet_counts)
display((100 * internet_shares).round(2).add_suffix(" (%)"))

px.bar(
    internet_shares.reset_index().melt(
        id_vars="cluster",
        var_name="internet_status",
        value_name="customer_share",
    ),
    x="cluster",
    y="customer_share",
    color="internet_status",
    barmode="stack",
    title="Internet-service composition within exploratory clusters",
).update_yaxes(tickformat=".0%").show()

,No internet,Has internet
cluster,,
0,20,1243
1,21,1597
2,17,1403
3,681,18


,No internet (%),Has internet (%)
cluster,,
0,1.58,98.42
1,1.30,98.70
2,1.20,98.80
3,97.42,2.58


## Summary and dashboard decisions

- Explored **700,000 training customers**. The highest-charge 10% contributed approximately **15.3% of total monthly charges**, indicating limited revenue concentration.
- Built interpretable **revenue × predicted-risk segments**, then overlaid campaign eligibility and capacity-limited profit targeting. High risk alone does not determine economic priority.
- Tested numeric-behavior clustering with **3–5 clusters**. Separation was weak (silhouette **0.11–0.13**) and stability varied. The four-cluster solution mainly reflected payment/support behavior and internet-service status: **97.4% of cluster 3 lacked internet service**.
- **Decision:** retain revenue and risk matrices, campaign overlays, and  behavioral/service filters for the dashboard. Clustering will stay only here.
